In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from itertools import chain
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import RandomizedSearchCV
import pandas as pd
from lightgbm import LGBMClassifier
import shap
import matplotlib.pyplot as plt

# Setup

In [ ]:
import sys
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Reading

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.getOrCreate()
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

In [ ]:
policy = spark.read.csv('fact_policy_churn.csv', header=True, inferSchema=True)
policy

# Exploratory

- Unique policies

In [ ]:
policy.select(count_distinct('policy_id'))

In [ ]:
policy.select(count_distinct('customer_id'))

In [ ]:
policy.select('policy_vrsn_no').distinct()

In [ ]:
window = Window.partitionBy('policy_id').orderBy(col('start_date').desc())

latest_policy = policy.withColumn('rn', row_number().over(window)) \
              .filter(col('rn') == 1) \
              .drop('rn')

In [ ]:
latest_policy.where(col('policy_status') == "Inactive").orderBy(col('end_date').desc())

- How many latest policies are active?

In [ ]:
latest_policy.where(col('policy_status') == "Active").count()

- How many latest policies are cancelled?

In [ ]:
latest_policy.where((col('cancellation_date').isNotNull())).count()

- How many historical latest policies where canceled?

In [ ]:
latest_policy.where((col('policy_status') == "Inactive") & (col('cancellation_date').isNotNull())).count()

- How many active policies will be canceled when they expire?

In [ ]:
latest_policy.where((col('policy_status') == "Active") & (col('cancellation_date').isNotNull())).count()

- How many latest policies have expired and haven't been renewed?

In [ ]:
latest_policy.where((col('policy_status') == "Inactive") & (col('cancellation_date').isNull())).count()

# Cleaning

In [ ]:
policy

In [ ]:
policy = policy.where(col('start_date') <= to_date(lit("2026-04-23"), "yyyy-MM-dd"))

- Set the policies with end_date later than the current_date to active

- To improve reproducability, we'll set the current_date to a fixed date, i.e. 23/04/2026

In [ ]:
policy = policy.withColumn('policy_status', when(col('end_date') >= to_date(lit("2026-04-23"), "yyyy-MM-dd"), "Active").otherwise("Inactive"))

In [ ]:
mapping = {"ΕΙΔΙΩΤ": "ΑΣΤΙΚΗ ΕΥΘΥΝΗ", "ΕΚΑΤΑ": "ΑΣΤΙΚΗ ΕΥΘΥΝΗ", "ΕΦΑΡΜ": "ΑΣΤΙΚΗ ΕΥΘΥΝΗ", "ΠΠΡΟΣ": "ΠΡΟΣΩΠΙΚΟ ΑΤΥΧΗΜΑ", "ΦΕΜΠΟ": "ΠΥΡΟΣ", "ΦΚΑΤΚ": "ΠΥΡΟΣ"}
mapping_expr = create_map([lit(x) for x in chain(*mapping.items())])
policy = policy.withColumn('lob', mapping_expr[col('product')])

In [ ]:
mapping = {"ΑΣΤΙΚΗ ΕΥΘΥΝΗ": "Ε", "ΠΡΟΣΩΠΙΚΟ ΑΤΥΧΗΜΑ": "Π", "ΠΥΡΟΣ": "Φ"}

mapping_expr = create_map([lit(x) for x in chain(*mapping.items())])

policy = policy.withColumn(
    "coverages",
    array_join(
        transform(
            split(col("coverages"), ","),
            lambda x: concat(mapping_expr[col("lob")], x)
        ),
        ","
    )
)

In [ ]:
policy

# Features

In [ ]:
round_features = {}

### Round 0

In [ ]:
round_features[0] = ['policy_vrsn_no', 'premium_amount']

### Round 1

In [ ]:
round_features[1] = ['total_premium_so_far', 'avg_premium_so_far', 'premium_amount_change', 
                    'premium_pct_change', 'no_of_coverages', 'avg_coverage_count_so_far', 
                    'coverage_changes', 'no_of_renewals_so_far', 'total_policies_per_customer_so_far', 
                    'customer_tenure_so_far', 'total_churns_per_customer_so_far', 'policy_tenure_so_far']

- [X]: flag_canceled, i
- [X]: flag_expired_but_not_renewed, i
- [X]: policy_is_churned, i
- [X]: total_premium_so_far, avg_premium_so_far
- [X]: premium_amount_change, i
- [X]: premium_pct_change, i
- [X]: no_of_coverages, i
- [X]: avg_coverage_count_so_far, i
- [X]: coverage_changes, i
- [X]: no_of_renewals_so_far, i
- [X]: total_policies_per_customer_so_far, i
- [X]: customer_tenure_so_far, i
- [X]: total_churns_per_customer_so_far, i
- [X]: policy_tenure_so_far, i

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [ ]:
policy

- []: flag_canceled
- []: flag_expired_but_not_renewed
- []: policy_is_churned

In [ ]:
latest_policy = latest_policy.withColumn('is_expired_and_not_renewed', when(col('policy_status') == "Inactive", True).otherwise(False))
latest_policy = latest_policy.withColumn('is_cancelled', when(col('cancellation_date').isNotNull(), True).otherwise(False))
policy = policy.join(latest_policy.select('policy_id', 'policy_vrsn_no', 'is_expired_and_not_renewed', 'is_cancelled'), on=['policy_id', 'policy_vrsn_no'], how='left')
policy = policy.fillna({'is_expired_and_not_renewed': False, 'is_cancelled': False})
policy = policy.withColumn('policy_is_churned', col('is_expired_and_not_renewed') | col('is_cancelled'))

- []: total_premium_so_far, avg_premium_so_far

In [ ]:
window = Window.partitionBy('policy_id').orderBy('policy_vrsn_no').rowsBetween(Window.unboundedPreceding, -1)
policy = policy.withColumn('avg_premium_so_far', avg('premium_amount').over(window))
policy = policy.withColumn('total_premium_so_far', sum('premium_amount').over(window))

- []: premium_amount_change
- []: premium_pct_change

In [ ]:
window = Window.partitionBy('policy_id').orderBy('policy_vrsn_no')
policy = policy.withColumn('prev_premium_amount', lag('premium_amount').over(window))
policy = policy.withColumn('premium_amount_change', when(col('prev_premium_amount').isNotNull(), round(col('premium_amount') - col('prev_premium_amount'), 2)).otherwise(None))
policy = policy.withColumn('premium_pct_change', when(col('prev_premium_amount').isNotNull(), col('premium_amount_change') / col('prev_premium_amount')).otherwise(None))
policy = policy.drop('prev_premium_amount')

- []: premium_change_squared

In [ ]:
policy = policy.withColumn('premium_change_squared', col('premium_amount_change') ** 2)

- []: no_of_coverages

In [ ]:
temp_policy = policy.withColumn('coverages_expl', explode(split(col('coverages'), ','))) \
    .groupBy('policy_id', 'policy_vrsn_no').count()

temp_policy = temp_policy.withColumnRenamed('count', 'no_of_coverages')

policy = policy.join(temp_policy, on=['policy_id', 'policy_vrsn_no'], how='inner')

- []: avg_coverage_count_so_far

In [ ]:
window = Window.partitionBy('policy_id').orderBy('policy_vrsn_no').rowsBetween(Window.unboundedPreceding, -1)
policy = policy.withColumn('avg_coverage_count_so_far', avg('no_of_coverages').over(window))

- []: coverage_changes

In [ ]:
window = Window.partitionBy('policy_id').orderBy('policy_vrsn_no')
policy = policy.withColumn('prev_no_of_coverages', lag('no_of_coverages').over(window))
policy = policy.withColumn('coverage_changes', when(col('prev_no_of_coverages').isNotNull(), col('no_of_coverages') - col('prev_no_of_coverages')).otherwise(None))
policy = policy.drop(col('prev_no_of_coverages'))

- []: no_of_renewals_so_far

In [ ]:
policy = policy.withColumn('no_of_renewals_so_far', col('policy_vrsn_no') - 1)

- []: total_policies_per_customer_so_far

In [ ]:
policy_level = (
    policy
    .groupBy('customer_id', 'policy_id')
    .agg(min('start_date').alias('policy_start_date'))
)

temp_policy = policy.join(
    policy_level,
    on=['customer_id', 'policy_id'],
    how='left'
)

df_alias = temp_policy.alias("current")
policy_alias = policy_level.alias("past")

result = df_alias.join(
    policy_alias,
    on=(
        (col("current.customer_id") == col("past.customer_id")) &
        (col("past.policy_start_date") < col("current.start_date"))
    ),
    how="left"
)

result = result.groupBy(
    "current.policy_id",
    "current.policy_vrsn_no",
    "current.customer_id",
    "current.start_date"
).agg(
    count_distinct("past.policy_id").alias("total_policies_per_customer_so_far")
)

temp_policy = temp_policy.join(
    result,
    on=['policy_id', 'policy_vrsn_no', 'customer_id', 'start_date'],
    how='left'
)

policy = temp_policy
policy = policy.drop(col('policy_start_date'))

- []: customer_tenure_so_far

In [ ]:
temp_policy = policy.groupBy('policy_id').agg(
    min('initial_start_date').alias('date_joined')
).select('policy_id', 'date_joined')

In [ ]:
policy = policy.join(temp_policy, on='policy_id', how='left')

In [ ]:
temp_customer = policy.groupBy('customer_id').agg(
    min('date_joined').alias('customer_date_joined')
).select('customer_id', 'customer_date_joined')
policy = policy.join(temp_customer, on='customer_id', how='left')

In [ ]:
policy = policy.withColumn('customer_tenure_so_far', date_diff(col('start_date'), col('customer_date_joined')))
policy = policy.drop(col('customer_date_joined'))
policy = policy.drop(col('date_joined'))

- []: total_churns_per_customer_so_far

In [ ]:
churn_events = (
    policy
    .filter(col('policy_is_churned') == True)
    .groupBy('customer_id', 'policy_id')
    .agg(
        max('start_date').alias('churn_date')  # last version start ≈ churn moment
    )
)

versions = policy.select(
    'customer_id',
    'policy_id',
    'policy_vrsn_no',
    'start_date'
)

v = versions.alias("v")
c = churn_events.alias("c")

joined = v.join(
    c,
    on=(
        (col("v.customer_id") == col("c.customer_id")) &
        (col("c.churn_date") < col("v.start_date"))
    ),
    how="left"
)

result = joined.groupBy(
    "v.customer_id",
    "v.policy_id",
    "v.policy_vrsn_no",
    "v.start_date"
).agg(
    count_distinct("c.policy_id").alias("total_churns_per_customer_so_far")
)

temp_policy = policy.join(
    result,
    on=['customer_id', 'policy_id', 'policy_vrsn_no', 'start_date'],
    how='left'
)

policy = temp_policy

In [ ]:
policy = policy.withColumn('policy_tenure_so_far', date_diff(col('start_date'), col('initial_start_date')))

### Round 2 Additions

In [ ]:
round_features[2] = ['log_premium', 'norm_premium',	
                    'norm_coverages', 'customer_churn_rate', 'premium_per_coverage',	
                    'policy_tenure_vs_customer_tenure']

- [X]: flag_large_premium_increase, i
- [X]: log_premium, i
- [X]: norm_premium, i
- [X]: norm_coverages, i
- [X]: customer_churn_rate, i (total_churns / total_policies)
- [X]: premium_per_coverage, i
- [X]: policy_tenure_vs_customer_tenure, i (policy_tenure / customer_tenure)

- []: flag_large_premium_increase

In [ ]:
policy = policy.withColumn('flag_large_premium_increase', when(col('premium_pct_change') > 0.25, True).otherwise(False))

- []: log_premium, norm_premium

In [ ]:
policy = policy.withColumn('log_premium', log(col('premium_amount')))
policy = policy.withColumn('norm_premium', when(col('avg_premium_so_far') != 0, col('premium_amount') / col('avg_premium_so_far')).otherwise(None))

- []: norm_coverages

In [ ]:
policy = policy.withColumn('norm_coverages', when(col('avg_coverage_count_so_far') != 0, col('no_of_coverages') / col('avg_coverage_count_so_far')).otherwise(None))

- []: customer_churn_rate

In [ ]:
policy = policy.withColumn('customer_churn_rate', when(col('total_policies_per_customer_so_far') != 0, 
                                                       col('total_churns_per_customer_so_far') / col('total_policies_per_customer_so_far'))
                                                       .otherwise(None))

- []: premium_per_coverage

In [ ]:
policy = policy.withColumn('premium_per_coverage', col('premium_amount') / col('no_of_coverages'))

- []: policy_tenure_vs_customer_tenure

In [ ]:
policy = policy.withColumn('policy_tenure_vs_customer_tenure', when(col('customer_tenure_so_far') != 0, 
                                                              col('policy_tenure_so_far') / col('customer_tenure_so_far'))
                                                              .otherwise(None))

### Round 3 Additions (Interactions)

In [ ]:
round_features[3] = ['premium_pressure', 'premium_increase_intensity', 'premium_vs_lifetime',
                    'price_sensitivity', 'premium_deviation_weighted', 'coverage_cost_intensity',
                    'coverage_price_impact', 'coverage_change_rate', 'coverage_instability',	
                    'churn_acceleration', 'policy_growth_rate', 'loyalty_risk_score',	
                    'instability_score', 'risk_amplification', 'burden_engagement_ratio']

- [X]: premium_pressure = premium_amount * policy_tenure_so_far
- [X]: premium_increase_intensity = premium_amount_change / (policy_tenure_so_far + 1)
- [X]: premium_vs_lifetime = premium_amount / (customer_tenure_so_far + 1)
- [X]: price_sensitivity = premium_pct_change * customer_churn_rate
- [X]: premium_deviation_weighted = (premium_amount - avg_premium_so_far) * customer_tenure_so_far
- [X]: premium_change_squared = premium_amount_change ** 2
- [X]: coverage_cost_intensity = premium_amount * no_of_coverages
- [X]: coverage_price_impact = coverage_changes * premium_amount_change
- [X]: coverage_change_rate = coverage_changes / (policy_tenure_so_far + 1)
- [X]: coverage_instability = coverage_changes * policy_tenure_so_far
- [X]: churn_acceleration = total_churns_per_customer_so_far / (customer_tenure_so_far + 1)
- [X]: policy_growth_rate = total_policies_per_customer_so_far / (customer_tenure_so_far + 1)
- [X]: loyalty_risk_score = customer_tenure_so_far * customer_churn_rate
- [X]: instability_score = abs(premium_amount_change) * coverage_changes
- [X]: risk_amplification = instability_score * policy_tenure_so_far
- [X]: burden_engagement_ratio = total_premium_so_far / (total_policies_per_customer_so_far + 1)

- []: premium_pressure

In [ ]:
policy = policy.withColumn('premium_pressure', col('premium_amount') * col('policy_tenure_so_far'))

- []: premium_increase_intensity = premium_amount_change / (policy_tenure_so_far + 1)

In [ ]:
policy = policy.withColumn('premium_increase_intensity', when(col('policy_tenure_so_far') != 0, 
                                                              col('premium_amount_change') / col('policy_tenure_so_far'))
                                                              .otherwise(None))

- []: premium_vs_lifetime = premium_amount / (customer_tenure_so_far + 1)

In [ ]:
policy = policy.withColumn('premium_vs_lifetime', when(col('customer_tenure_so_far') != 0, 
                                                       col('premium_amount') / col('customer_tenure_so_far'))
                                                       .otherwise(col('premium_amount')))

- []: price_sensitivity = premium_pct_change * customer_churn_rate

In [ ]:
policy = policy.withColumn('price_sensitivity', col('premium_pct_change') * col('customer_churn_rate'))

- []: premium_deviation_weighted = (premium_amount - avg_premium_so_far) * customer_tenure_so_far

In [ ]:
policy = policy.withColumn('premium_deviation_weighted', (col('premium_amount') - col('avg_premium_so_far')) * col('customer_tenure_so_far'))

- []: coverage_cost_intensity = premium_amount * no_of_coverages

In [ ]:
policy = policy.withColumn('coverage_cost_intensity', col('premium_amount') * col('no_of_coverages'))

- []: coverage_price_impact = coverage_changes * premium_amount_change

In [ ]:
policy = policy.withColumn('coverage_price_impact', col('coverage_changes') * col('premium_amount_change'))

- []: coverage_change_rate = coverage_changes / (policy_tenure_so_far + 1)

In [ ]:
policy = policy.withColumn('coverage_change_rate', when(col('policy_tenure_so_far') != 0, 
                                                        col('coverage_changes') / col('policy_tenure_so_far'))
                                                        .otherwise(None))

- []: coverage_instability = coverage_changes * policy_tenure_so_far

In [ ]:
policy = policy.withColumn('coverage_instability', col('coverage_changes') * col('policy_tenure_so_far'))

- []: churn_acceleration = total_churns_per_customer_so_far / (customer_tenure_so_far + 1)

In [ ]:
policy = policy.withColumn('churn_acceleration', when(col('customer_tenure_so_far') != 0, 
                                                        col('total_churns_per_customer_so_far') / col('customer_tenure_so_far'))
                                                        .otherwise(None))

- []: policy_growth_rate = total_policies_per_customer_so_far / (customer_tenure_so_far + 1)

In [ ]:
policy = policy.withColumn('policy_growth_rate', when(col('customer_tenure_so_far') != 0, 
                                                        col('total_policies_per_customer_so_far') / col('customer_tenure_so_far'))
                                                        .otherwise(None))

- []: loyalty_risk_score = customer_tenure_so_far * customer_churn_rate

In [ ]:
policy = policy.withColumn('loyalty_risk_score', col('customer_tenure_so_far') * col('customer_churn_rate'))

- []: instability_score = abs(premium_amount_change) * coverage_changes

In [ ]:
policy = policy.withColumn('instability_score', abs(col('premium_amount_change')) * col('coverage_changes'))

- []: risk_amplification = instability_score * policy_tenure_so_far

In [ ]:
policy = policy.withColumn('risk_amplification', col('instability_score') * col('policy_tenure_so_far'))

- []: burden_engagement_ratio = total_premium_so_far / (total_policies_per_customer_so_far + 1)

In [ ]:
policy = policy.withColumn('burden_engagement_ratio', when(col('total_policies_per_customer_so_far') != 0, 
                                                        col('total_premium_so_far') / col('total_policies_per_customer_so_far'))
                                                        .otherwise(None))

### Final Look

- Null Percentages in features

In [ ]:
# 1) Aggregate counts into a single row
agg_exprs = []
for c in policy.columns:
    agg_exprs.append(count(when(col(c).isNull(), 1)).alias(f"{c}_nulls"))
    agg_exprs.append(count(when(col(c).isNotNull(), 1)).alias(f"{c}_non_nulls"))

agg_df = policy.agg(*agg_exprs)

# 2) Build stack expression to unpivot
n = len(policy.columns)
stack_expr = "stack({}, {}) as (column, nulls, non_nulls)".format(
    n,
    ", ".join(
        [f"'{c}', `{c}_nulls`, `{c}_non_nulls`" for c in policy.columns]
    )
)

# 3) Final result
result = agg_df.selectExpr(stack_expr)
result

In [ ]:
policy = policy.fillna(0)

In [ ]:
policy

# Correlation Analysis

In [ ]:
undroppable_features = ['policy_tenure_so_far']

In [ ]:
CORRELATION_THRESHOLD = 0.85

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import numpy as np

features = []

for i in range(len(round_features) - 1, -1, -1):
    features.extend(round_features[i])

# Assemble features into a vector
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_vector = assembler.transform(policy).select("features")

# Compute correlation matrix
corr_matrix = Correlation.corr(df_vector, "features").head()[0]

matrix = np.array(corr_matrix.toArray())
matrix_df = pd.DataFrame(matrix)
matrix_df.columns = features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(25, 25))
sns.heatmap(matrix_df, annot=True, cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
rows, cols = np.where(matrix_df > CORRELATION_THRESHOLD)

appearance_counts = {}

for feat in features:
    appearance_counts[feat] = [0, 0, 0]

for row, col_ in zip(rows, cols):
    if row < col_:
        appearance_counts[features[row]][0] += 1
        appearance_counts[features[row]][1] += matrix_df.iloc[row, col_]
        appearance_counts[features[col_]][0] += 1
        appearance_counts[features[col_]][1] += matrix_df.iloc[row, col_]

In [ ]:
for feat in features:
    appearance_counts[feat][2] = appearance_counts[feat][1] / (appearance_counts[feat][0] + 1e-5)

In [ ]:
dropped_features = {}

for i in range(len(round_features) - 1, -1, -1):
    dropped_features[i] = []
    print('\n')
    print(f'Round {i} of features \n')

    for feat in round_features[i]:   
        if appearance_counts[feat][2] > CORRELATION_THRESHOLD:
            print(feat, appearance_counts[feat][2])
            dropped_features[i].append(feat)

    for feat in dropped_features[i]:
        if feat not in undroppable_features:
            features.remove(feat)
                
    # Assemble features into a vector
    assembler = VectorAssembler(inputCols=features, outputCol="features")
    df_vector = assembler.transform(policy).select("features")

    # Compute correlation matrix
    corr_matrix = Correlation.corr(df_vector, "features").head()[0]

    matrix = np.array(corr_matrix.toArray())
    matrix_df = pd.DataFrame(matrix)
    matrix_df.columns = features   

    rows, cols = np.where(matrix_df > CORRELATION_THRESHOLD)

    appearance_counts = {}

    for feat in features:
        appearance_counts[feat] = [0, 0, 0]

    for row, col_ in zip(rows, cols):
        if row < col_:
            appearance_counts[features[row]][0] += 1
            appearance_counts[features[row]][1] += matrix_df.iloc[row, col_]
            appearance_counts[features[col_]][0] += 1
            appearance_counts[features[col_]][1] += matrix_df.iloc[row, col_]

    for feat in features:
        appearance_counts[feat][2] = appearance_counts[feat][1] / (appearance_counts[feat][0] + 1e-5)

In [ ]:
features

In [ ]:
for i in range(len(dropped_features) - 1, -1, -1):
    print(dropped_features[i])

- Verification Correlation

In [ ]:
# Assemble features into a vector
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_vector = assembler.transform(policy).select("features")

# Compute correlation matrix
corr_matrix = Correlation.corr(df_vector, "features").head()[0]

matrix = np.array(corr_matrix.toArray())
matrix_df = pd.DataFrame(matrix)
matrix_df.columns = features

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(25, 25))
sns.heatmap(matrix_df, annot=True, cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
rows, cols = np.where(matrix_df > 0.80)

for row, col_ in zip(rows, cols):
    if row < col_:
        print(features[row], features[col_], matrix_df.iloc[row, col_])

# Models

### Data Specification

In [ ]:
spark_df = policy
id_col = 'policy_id'
event_col = 'policy_is_churned'
duration_col = 'policy_tenure_so_far'
start_date_col = 'start_date'
end_date_col = 'end_date'

In [ ]:
model_columns = [id_col, event_col, duration_col, start_date_col, end_date_col]

num_cols = features

for feat in features:
    if feat in model_columns:
        num_cols.remove(feat)

In [ ]:
cat_cols = [
    'premium_frequency',
    'product',
    'lob', 
    'flag_large_premium_increase'
]

## Classic Models

### Data Preparation

In [ ]:
classic_df = spark_df.select(id_col, event_col, start_date_col, *num_cols, *cat_cols).toPandas()

In [ ]:
classic_df.sort_values(by=[id_col, start_date_col])

In [ ]:
classic_df.drop(columns=[start_date_col], inplace=True)

In [ ]:
dummy_df = pd.get_dummies(classic_df, columns=cat_cols, drop_first=True)
dummy_cols = dummy_df.columns.difference(classic_df.columns).to_list()
classic_df = dummy_df

In [ ]:
features = num_cols + dummy_cols

- Creating train-test splits with similar churn distributions

In [ ]:
distribution_df = classic_df.groupby(id_col).agg(
    {event_col: 'last'}
).reset_index()

no_of_churned_subjects = distribution_df.loc[distribution_df[event_col] == True].shape[0]
no_of_non_churned_subjects = distribution_df.loc[distribution_df[event_col] == False].shape[0]
no_of_total_subjects = no_of_churned_subjects + no_of_non_churned_subjects
churned_ratio = no_of_churned_subjects / no_of_total_subjects

print('Total churn ratio', churned_ratio, no_of_total_subjects)

In [ ]:
churned_ids = distribution_df.loc[distribution_df[event_col] == True, id_col].to_list()
non_churned_ids = distribution_df.loc[distribution_df[event_col] == False, id_col].to_list()

In [ ]:
from sklearn.model_selection import train_test_split

train_churned, test_churned = train_test_split(churned_ids, test_size=0.2, random_state=42)
train_non_churned, test_non_churned = train_test_split(non_churned_ids, test_size=0.2, random_state=42)

In [ ]:
train_df = classic_df.loc[classic_df[id_col].isin(train_churned + train_non_churned)]
test_df = classic_df.loc[classic_df[id_col].isin(test_churned + test_non_churned)]

In [ ]:
distribution_df = train_df.groupby(id_col).agg(
    {event_col: 'last'}
).reset_index()

no_of_churned_subjects = distribution_df.loc[distribution_df[event_col] == True].shape[0]
no_of_non_churned_subjects = distribution_df.loc[distribution_df[event_col] == False].shape[0]
no_of_total_subjects = no_of_churned_subjects + no_of_non_churned_subjects
churned_ratio = no_of_churned_subjects / no_of_total_subjects

print('Training churn ratio', churned_ratio, no_of_total_subjects)

In [ ]:
distribution_df = test_df.groupby(id_col).agg(
    {event_col: 'last'}
).reset_index()

no_of_churned_subjects = distribution_df.loc[distribution_df[event_col] == True].shape[0]
no_of_non_churned_subjects = distribution_df.loc[distribution_df[event_col] == False].shape[0]
no_of_total_subjects = no_of_churned_subjects + no_of_non_churned_subjects
churned_ratio = no_of_churned_subjects / no_of_total_subjects

print('Testing churn ratio', churned_ratio, no_of_total_subjects)

In [ ]:
results = pd.DataFrame(columns=['Model', 'ROC AUC', 'PR AUC'])

In [ ]:
train_distribution = train_df.groupby(id_col).agg({event_col: "last"}).reset_index()

n_pos = (train_distribution[event_col] == True).sum()
n_neg = (train_distribution[event_col] == False).sum()

scale_pos_weight = n_neg / n_pos

print("Train churn ratio:", n_pos / (n_pos + n_neg))
print("scale_pos_weight:", scale_pos_weight)

#### Method for imbalance checking

In [ ]:
def handle_imbalance(X_train, y_train, threshold=0.85):
    import pandas as pd
    from sklearn.utils import resample

    df = X_train.copy()
    df["target"] = y_train.values if hasattr(y_train, "values") else y_train

    ratios = df["target"].value_counts(normalize=True)
    majority_ratio = ratios.max()

    print("Class ratios:")
    print(ratios)

    if majority_ratio < threshold:
        print("\nNo strong imbalance → skip balancing")
        return X_train.copy(), y_train.copy()

    print("\nStrong imbalance detected → apply undersampling")

    majority_class = ratios.idxmax()
    minority_class = ratios.idxmin()

    df_majority = df[df["target"] == majority_class]
    df_minority = df[df["target"] == minority_class]

    df_majority_down = resample(
        df_majority,
        replace=False,
        n_samples=len(df_minority),
        random_state=42
    )

    df_balanced = pd.concat([df_majority_down, df_minority])
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    X_bal = df_balanced.drop(columns=["target"])
    y_bal = df_balanced["target"]

    print("\nBalanced class ratios:")
    print(y_bal.value_counts(normalize=True))

    return X_bal, y_bal

#### Method for top features

In [ ]:
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score

def evaluate_top_features(model, X_train, y_train, X_test, y_test, top_features, model_name):
    X_train_top = X_train[top_features].copy()
    X_test_top = X_test[top_features].copy()

    model_top = clone(model)
    model_top.fit(X_train_top, y_train)

    y_proba = model_top.predict_proba(X_test_top)[:, 1]

    return {
        "Model": model_name,
        "ROC AUC": roc_auc_score(y_test, y_proba),
        "PR AUC": average_precision_score(y_test, y_proba),
        "model_obj": model_top,
        "features": top_features,
        "y_proba": y_proba,
        "X_test_top": X_test_top,
        "X_train_top": X_train_top
    }

### XGBoost

In [ ]:
xgb_train_df = train_df.copy()
xgb_test_df = test_df.copy()

In [ ]:
xgb_X_train = xgb_train_df.drop(columns=[id_col, event_col])
xgb_y_train = xgb_train_df[event_col]

In [ ]:
xgb_X_test = xgb_test_df.drop(columns=[id_col, event_col])
xgb_y_test = xgb_test_df[event_col]

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

#### Initial Training

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier


xgb_default = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb_default.fit(xgb_X_train, xgb_y_train)

y_proba_xgb = xgb_default.predict_proba(xgb_X_test)[:, 1]

print("Test ROC AUC:", roc_auc_score(xgb_y_test, y_proba_xgb))
print("Test PR AUC:", average_precision_score(xgb_y_test, y_proba_xgb))

In [ ]:
results = pd.concat([results, pd.DataFrame([['XGB Initial', roc_auc_score(xgb_y_test, y_proba_xgb), average_precision_score(xgb_y_test, y_proba_xgb)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

#### Hyperparameter Tuning

In [ ]:
scale_pos_weight = (len(xgb_y_train[xgb_y_train == 0]) / len(xgb_y_train[xgb_y_train == 1]))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

param_grid = {
    'n_estimators': [300, 500, 800],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.03, 0.05],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3]
}

model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)

random_search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=20,
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=-1, 
    random_state=42
)

random_search.fit(xgb_X_train, xgb_y_train)

best_model_xgb = random_search.best_estimator_
y_proba_xgb = best_model_xgb.predict_proba(xgb_X_test)[:, 1]

print("Test ROC AUC:", roc_auc_score(xgb_y_test, y_proba_xgb))
print("Test PR AUC:", average_precision_score(xgb_y_test, y_proba_xgb))

In [ ]:
results = pd.concat([results, pd.DataFrame([['XGB HyperParameter', roc_auc_score(xgb_y_test, y_proba_xgb), average_precision_score(xgb_y_test, y_proba_xgb)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
xgb_X_test_best = xgb_X_test.copy()
xgb_X_train_best = xgb_X_train.copy()

#### Removing unimportant features

In [ ]:
feat_importance = pd.Series(
    best_model_xgb.feature_importances_,
    index=xgb_X_train.columns
)

In [ ]:
low_importance = feat_importance[feat_importance < 0.005].index

xgb_X_train = xgb_X_train_best.drop(columns=low_importance, errors="ignore")
xgb_X_test = xgb_X_test_best.drop(columns=low_importance, errors="ignore")

In [ ]:
best_model_xgb.fit(xgb_X_train, xgb_y_train)

y_proba_xgb = best_model_xgb.predict_proba(xgb_X_test)[:, 1]

print("Test ROC AUC:", roc_auc_score(xgb_y_test, y_proba_xgb))
print("Test PR AUC:", average_precision_score(xgb_y_test, y_proba_xgb))

In [ ]:
results = pd.concat([results, pd.DataFrame([['XGB WithoutUnimportant', roc_auc_score(xgb_y_test, y_proba_xgb), average_precision_score(xgb_y_test, y_proba_xgb)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
xgb_X_test_best = xgb_X_test.copy()
xgb_X_train_best = xgb_X_train.copy()

### Keeping only the top 20 features

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
best_model_xgb.fit(xgb_X_train_best, xgb_y_train);

In [ ]:
feat_imp = pd.DataFrame({
    "feature": xgb_X_train.columns,
    "importance": best_model_xgb.feature_importances_
})

In [ ]:
top20_xgb = feat_imp.sort_values("importance", ascending=False)["feature"].head(20).tolist()

xgb_top20_result = evaluate_top_features(
    best_model_xgb,
    xgb_X_train_best,
    xgb_y_train,
    xgb_X_test_best,
    xgb_y_test,
    top20_xgb,
    "XGB Top 20"
)

print(
    xgb_top20_result["Model"],
    xgb_top20_result["ROC AUC"],
    xgb_top20_result["PR AUC"]
)

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame([xgb_top20_result])[["Model", "ROC AUC", "PR AUC"]]
], ignore_index=True)

#### SHAP

In [ ]:
best_model_xgb.fit(xgb_X_train_best, xgb_y_train)
y_proba_xgb = best_model_xgb.predict_proba(xgb_X_test_best)[:, 1]

print("Test ROC AUC:", roc_auc_score(xgb_y_test, y_proba_xgb))
print("Test PR AUC:", average_precision_score(xgb_y_test, y_proba_xgb))

In [ ]:
import shap
import numpy as np
import pandas as pd

import warnings

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="shap"
)

explainer = shap.TreeExplainer(best_model_xgb)

shap_values = explainer.shap_values(xgb_X_test_best)

shap.summary_plot(shap_values, xgb_X_test_best, plot_type="bar")

shap_values = explainer.shap_values(xgb_X_test_best)

shap.summary_plot(shap_values, xgb_X_test_best)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "feature": xgb_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    xgb_X_test_best.iloc[0]
)

### No scale_pos_weight

In [ ]:
xgb_no_weight = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb_search_no_weight = RandomizedSearchCV(
    estimator=xgb_no_weight,
    param_distributions=param_grid,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

xgb_search_no_weight.fit(xgb_X_train_best, xgb_y_train)

best_model_xgb_no_weight = xgb_search_no_weight.best_estimator_
y_proba_no_weight = best_model_xgb_no_weight.predict_proba(xgb_X_test_best)[:, 1]

roc_no = roc_auc_score(xgb_y_test, y_proba_no_weight)
pr_no = average_precision_score(xgb_y_test, y_proba_no_weight)

print("NO WEIGHT ROC AUC:", roc_no)
print("NO WEIGHT PR AUC:", pr_no)

In [ ]:
results = pd.concat([results, pd.DataFrame([['XGB No Weight', roc_auc_score(xgb_y_test, y_proba_no_weight), average_precision_score(xgb_y_test, y_proba_no_weight)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Results

In [ ]:
results

#### SHAP

In [ ]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(best_model_xgb_no_weight)

shap_values = explainer.shap_values(xgb_X_test_best)

shap.summary_plot(shap_values, xgb_X_test_best, plot_type="bar")

shap_values = explainer.shap_values(xgb_X_test_best)

shap.summary_plot(shap_values, xgb_X_test_best)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "feature": xgb_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

i = 20

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[i],
    xgb_X_test_best.iloc[i]
)

### Check for imbalance

In [ ]:
xgb_X_train_bal, xgb_y_train_bal = handle_imbalance(
    xgb_X_train_best,
    xgb_y_train,
    threshold=0.85
)

In [ ]:
best_model_xgb.fit(xgb_X_train_bal, xgb_y_train_bal)

y_proba_xgb = best_model_xgb.predict_proba(xgb_X_test_best)[:, 1]

print("Test ROC AUC:", roc_auc_score(xgb_y_test, y_proba_xgb))
print("Test PR AUC:", average_precision_score(xgb_y_test, y_proba_xgb))

### Threshold Analysis

In [ ]:
xgb_search_no_weight.fit(xgb_X_train_best, xgb_y_train)

best_model_xgb_no_weight = xgb_search_no_weight.best_estimator_
y_proba_no_weight = best_model_xgb_no_weight.predict_proba(xgb_X_test_best)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for t in thresholds:
    y_pred = (y_proba_no_weight >= t).astype(int)

    rows.append({
        "Threshold": t,
        "Precision": precision_score(xgb_y_test, y_pred),
        "Recall": recall_score(xgb_y_test, y_pred),
        "F1": f1_score(xgb_y_test, y_pred)
    })

threshold_df_xgb = pd.DataFrame(rows)
threshold_df_xgb

### Logistic Regression

In [ ]:
lr_train_df = train_df.copy()
lr_test_df = test_df.copy()

In [ ]:
lr_X_train = lr_train_df.drop(columns=[id_col, event_col])
lr_y_train = lr_train_df[event_col]

In [ ]:
lr_X_test = lr_test_df.drop(columns=[id_col, event_col])
lr_y_test = lr_test_df[event_col]

### Initial Training

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

log_default = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

log_default.fit(lr_X_train, lr_y_train)

y_proba_lr = log_default.predict_proba(lr_X_test)[:, 1]

print("LOG Test ROC AUC:", roc_auc_score(lr_y_test, y_proba_lr))
print("LOG Test PR AUC:", average_precision_score(lr_y_test, y_proba_lr))

In [ ]:
results = pd.concat([results, pd.DataFrame([['LR Initial', roc_auc_score(lr_y_test, y_proba_lr), average_precision_score(lr_y_test, y_proba_lr)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Hyperparameter Tuning

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

log_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=42
    ))
])

log_param_grid = [
    # L2 regularization
    {
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs", "liblinear", "saga"],
        "model__C": np.logspace(-4, 2, 20),
        "model__class_weight": [None, "balanced"]
    },

    # L1 regularization
    {
        "model__penalty": ["l1"],
        "model__solver": ["liblinear", "saga"],
        "model__C": np.logspace(-4, 1, 15),
        "model__class_weight": [None, "balanced"]
    },

    # ElasticNet - only saga supports it
    {
        "model__penalty": ["elasticnet"],
        "model__solver": ["saga"],
        "model__C": np.logspace(-4, 1, 15),
        "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
        "model__class_weight": [None, "balanced"]
    }
]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

random_search_log = RandomizedSearchCV(
    estimator=log_pipe,
    param_distributions=log_param_grid,
    n_iter=60,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

random_search_log.fit(lr_X_train, lr_y_train)

best_model_log = random_search_log.best_estimator_

y_proba_log = best_model_log.predict_proba(lr_X_test)[:, 1]

print("LOG Best params:", random_search_log.best_params_)
print("LOG Best CV ROC AUC:", random_search_log.best_score_)
print("LOG Test ROC AUC:", roc_auc_score(lr_y_test, y_proba_log))
print("LOG Test PR AUC:", average_precision_score(lr_y_test, y_proba_log))

In [ ]:
results = pd.concat([results, pd.DataFrame([['LR HyperParameter', roc_auc_score(lr_y_test, y_proba_log), average_precision_score(lr_y_test, y_proba_log)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
lr_X_train_best = lr_X_train.copy()
lr_X_test_best = lr_X_test.copy()

### Removing unimportant features

In [ ]:
coef_df = pd.DataFrame({
    "feature": lr_X_train.columns,
    "coefficient": best_model_log.named_steps["model"].coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

low_coef = coef_df[coef_df["abs_coefficient"] < 0.005]["feature"]

In [ ]:
lr_X_train = lr_X_train_best.drop(columns=low_coef)
lr_X_test  = lr_X_test_best.drop(columns=low_coef)

best_model_log.fit(lr_X_train, lr_y_train)

y_proba_log = best_model_log.predict_proba(lr_X_test)[:, 1]

print("LOG Test ROC AUC:", roc_auc_score(lr_y_test, y_proba_log))
print("LOG Test PR AUC:", average_precision_score(lr_y_test, y_proba_log))

In [ ]:
results = pd.concat([results, pd.DataFrame([['LR WithoutUnimportant', roc_auc_score(lr_y_test, y_proba_log), average_precision_score(lr_y_test, y_proba_log)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Keeping only the top 20 features

In [ ]:
best_model_log.fit(lr_X_train_best, lr_y_train);

In [ ]:
top20_lr = coef_df.sort_values("abs_coefficient", ascending=False)["feature"].head(20).tolist()

lr_top20_result = evaluate_top_features(
    best_model_log,
    lr_X_train_best,
    lr_y_train,
    lr_X_test_best,
    lr_y_test,
    top20_lr,
    "LR Top 20"
)

print(
    lr_top20_result["Model"],
    lr_top20_result["ROC AUC"],
    lr_top20_result["PR AUC"]
)

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame([lr_top20_result])[["Model", "ROC AUC", "PR AUC"]]
], ignore_index=True)

### Results

In [ ]:
results

#### SHAP

In [ ]:
best_model_log.fit(lr_X_train_best, lr_y_train)

y_proba_log = best_model_log.predict_proba(lr_X_test_best)[:, 1]

print("LOG Test ROC AUC:", roc_auc_score(lr_y_test, y_proba_log))
print("LOG Test PR AUC:", average_precision_score(lr_y_test, y_proba_log))

In [ ]:
import shap
import numpy as np
import pandas as pd

model = best_model_log.named_steps["model"]
scaler = best_model_log.named_steps["scaler"]

X_train_scaled = scaler.transform(lr_X_train_best)
X_test_scaled = scaler.transform(lr_X_test_best)

explainer = shap.LinearExplainer(model, X_train_scaled)

shap_values = explainer.shap_values(X_test_scaled)

shap.summary_plot(shap_values, lr_X_test_best, plot_type="bar")

shap.summary_plot(shap_values, lr_X_test_best)

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_importance = pd.DataFrame({
    "feature": lr_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values[0],
    lr_X_test_best.iloc[0]
)

### Check for Imbalance

In [ ]:
lr_X_train_bal, lr_y_train_bal = handle_imbalance(lr_X_train_best, lr_y_train)

In [ ]:
best_model_log.fit(lr_X_train_bal, lr_y_train_bal)

y_proba_log = best_model_log.predict_proba(lr_X_test_best)[:, 1]

print("LOG Test ROC AUC:", roc_auc_score(lr_y_test, y_proba_log))
print("LOG Test PR AUC:", average_precision_score(lr_y_test, y_proba_log))

### Threshold Analysis

In [ ]:
best_model_log.fit(lr_X_train_best, lr_y_train)

y_proba_log = best_model_log.predict_proba(lr_X_test_best)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for t in thresholds:
    y_pred = (y_proba_log >= t).astype(int)

    rows.append({
        "Threshold": t,
        "Precision": precision_score(lr_y_test, y_pred),
        "Recall": recall_score(lr_y_test, y_pred),
        "F1": f1_score(lr_y_test, y_pred)
    })

threshold_df_lr = pd.DataFrame(rows)
threshold_df_lr

### Coefficients

In [ ]:
import pandas as pd
import numpy as np

# If your LR model is inside a pipeline
lr_model = best_model_log.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": lr_X_train_best.columns,
    "coefficient": lr_model.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

coef_df.head(20)

### Random Forest

In [ ]:
rf_train_df = train_df.copy()
rf_test_df = test_df.copy()

In [ ]:
rf_X_train = rf_train_df.drop(columns=[id_col, event_col])
rf_y_train = rf_train_df[event_col]

In [ ]:
rf_X_test = rf_test_df.drop(columns=[id_col, event_col])
rf_y_test = rf_test_df[event_col]

### Initial Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_default = RandomForestClassifier(random_state=42)

rf_default.fit(rf_X_train, rf_y_train)

rf_proba = rf_default.predict_proba(rf_X_test)[:, 1]

print("RF Test ROC AUC:", roc_auc_score(rf_y_test, rf_proba))
print("RF Test PR AUC:", average_precision_score(rf_y_test, rf_proba))

In [ ]:
results = pd.concat([results, pd.DataFrame([['RF Initial', roc_auc_score(rf_y_test, rf_proba), average_precision_score(rf_y_test, rf_proba)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_param_grid_v2 = {
    "n_estimators": [400, 600, 800, 1000, 1200],

    "max_depth": [None, 4, 6, 8, 10, 12],

    "min_samples_split": [2, 5, 10, 20, 40],
    "min_samples_leaf": [1, 2, 4, 8, 12],

    "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7],

    "bootstrap": [True],
    "max_samples": [0.6, 0.8, 1.0],

    "class_weight": [None, "balanced", "balanced_subsample"],

    "min_impurity_decrease": [0.0, 0.0001, 0.001, 0.005],
    "ccp_alpha": [0.0, 0.0001, 0.001, 0.005]
}

rf_search_roc = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_grid_v2,
    n_iter=60,
    scoring="roc_auc",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

rf_search_roc.fit(rf_X_train, rf_y_train)

best_model_rf = rf_search_roc.best_estimator_
rf_proba_roc = best_model_rf.predict_proba(rf_X_test)[:, 1]

print("RF ROC Best params:", rf_search_roc.best_params_)
print("RF ROC Best CV ROC AUC:", rf_search_roc.best_score_)
print("RF ROC Test ROC AUC:", roc_auc_score(rf_y_test, rf_proba_roc))
print("RF ROC Test PR AUC:", average_precision_score(rf_y_test, rf_proba_roc))

In [ ]:
results = pd.concat([results, pd.DataFrame([['RF HyperParameter', roc_auc_score(rf_y_test, rf_proba_roc), average_precision_score(rf_y_test, rf_proba_roc)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
rf_X_test_best = rf_X_test.copy()
rf_X_train_best = rf_X_train.copy()

### Removing unimportant features

In [ ]:
rf_importance = pd.DataFrame({
    "feature": rf_X_train_best.columns,
    "importance": best_model_rf.feature_importances_
})

rf_importance = rf_importance.sort_values("importance", ascending=False)

In [ ]:
low_importance_rf = rf_importance[
    rf_importance["importance"] < 0.005
]["feature"]

In [ ]:
rf_X_train = rf_X_train_best.drop(columns=low_importance_rf, errors='ignore')
rf_X_test  = rf_X_test_best.drop(columns=low_importance_rf, errors='ignore')

best_model_rf.fit(rf_X_train, rf_y_train)

y_proba_rf = best_model_rf.predict_proba(rf_X_test)[:, 1]

print("RF Test ROC AUC:", roc_auc_score(rf_y_test, y_proba_rf))
print("RF Test PR AUC:", average_precision_score(rf_y_test, y_proba_rf))

In [ ]:
results = pd.concat([results, pd.DataFrame([['RF Without Unimportant', roc_auc_score(rf_y_test, y_proba_rf), average_precision_score(rf_y_test, y_proba_rf)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
rf_X_test_best = rf_X_test.copy()
rf_X_train_best = rf_X_train.copy()

### Keeping only the top 20 features

In [ ]:
best_model_rf.fit(rf_X_train_best, rf_y_train);

In [ ]:
rf_importance = pd.DataFrame({
    "feature": rf_X_train_best.columns,
    "importance": best_model_rf.feature_importances_
})

In [ ]:
top20_rf = rf_importance.sort_values("importance", ascending=False)["feature"].head(20).tolist()

rf_top20_result = evaluate_top_features(
    best_model_rf,
    rf_X_train_best,
    rf_y_train,
    rf_X_test_best,
    rf_y_test,
    top20_rf,
    "RF Top 20"
)

print(
    rf_top20_result["Model"],
    rf_top20_result["ROC AUC"],
    rf_top20_result["PR AUC"]
)

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame([rf_top20_result])[["Model", "ROC AUC", "PR AUC"]]
], ignore_index=True)

### Results

In [ ]:
results

#### SHAP

In [ ]:
best_model_rf.fit(rf_X_train_best, rf_y_train)

y_proba_rf = best_model_rf.predict_proba(rf_X_test_best)[:, 1]

print("RF Test ROC AUC:", roc_auc_score(rf_y_test, y_proba_rf))
print("RF Test PR AUC:", average_precision_score(rf_y_test, y_proba_rf))

In [ ]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(best_model_rf)
shap_exp = explainer(rf_X_test_best)

# binary classification handling
if len(shap_exp.values.shape) == 3:
    shap_values_plot = shap_exp.values[:, :, 1]
    expected_value = shap_exp.base_values[:, 1].mean()
else:
    shap_values_plot = shap_exp.values
    expected_value = np.mean(shap_exp.base_values)

shap.summary_plot(shap_values_plot, rf_X_test_best, plot_type="bar")
shap.summary_plot(shap_values_plot, rf_X_test_best)

mean_abs_shap = np.abs(shap_values_plot).mean(axis=0)

shap_importance = pd.DataFrame({
    "feature": rf_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

shap.initjs()
shap.force_plot(
    expected_value,
    shap_values_plot[0],
    rf_X_test_best.iloc[0]
)

### Check for Imbalance

In [ ]:
rf_X_train_bal, rf_y_train_bal = handle_imbalance(rf_X_train_best, rf_y_train)

In [ ]:
best_model_rf.fit(rf_X_train_bal, rf_y_train_bal)
y_proba_rf = best_model_rf.predict_proba(rf_X_test_best)[:, 1]

print("RF Test ROC AUC:", roc_auc_score(rf_y_test, y_proba_rf))
print("RF Test PR AUC:", average_precision_score(rf_y_test, y_proba_rf))

### Threshold Analysis

In [ ]:
best_model_rf.fit(rf_X_train_best, rf_y_train)
y_proba_rf = best_model_rf.predict_proba(rf_X_test_best)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for t in thresholds:
    y_pred = (y_proba_rf >= t).astype(int)

    rows.append({
        "Threshold": t,
        "Precision": precision_score(rf_y_test, y_pred),
        "Recall": recall_score(rf_y_test, y_pred),
        "F1": f1_score(rf_y_test, y_pred)
    })

threshold_df_rf = pd.DataFrame(rows)
threshold_df_rf

### HistGradientBoosting

In [ ]:
hgb_train_df = train_df.copy()
hgb_test_df = test_df.copy()

In [ ]:
hgb_X_train = hgb_train_df.drop(columns=[id_col, event_col])
hgb_y_train = hgb_train_df[event_col]

In [ ]:
hgb_X_test = hgb_test_df.drop(columns=[id_col, event_col])
hgb_y_test = hgb_test_df[event_col]

### Initial Training

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_default = HistGradientBoostingClassifier(
    random_state=42
)

hgb_default.fit(hgb_X_train, hgb_y_train)

hgb_proba = hgb_default.predict_proba(hgb_X_test)[:, 1]

print("HGB Test ROC AUC:", roc_auc_score(hgb_y_test, hgb_proba))
print("HGB Test PR AUC:", average_precision_score(hgb_y_test, hgb_proba))

In [ ]:
results = pd.concat([results, pd.DataFrame([['HGB Initial', roc_auc_score(hgb_y_test, hgb_proba), average_precision_score(hgb_y_test, hgb_proba)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb = HistGradientBoostingClassifier(random_state=42)

hgb_param_grid_v2 = {
    "learning_rate": [0.003, 0.005, 0.01, 0.03, 0.05],
    "max_iter": [200, 400, 600, 800],
    "max_depth": [2, 3, 5, 7, None],
    "max_leaf_nodes": [7, 15, 31, 63],
    "min_samples_leaf": [5, 10, 20, 50, 100],
    "l2_regularization": [0.0, 0.01, 0.1, 1.0, 5.0, 10.0],
    "max_features": [0.5, 0.7, 0.9, 1.0]
}

hgb_search = RandomizedSearchCV(
    estimator=hgb,
    param_distributions=hgb_param_grid_v2,
    n_iter=25,
    scoring="roc_auc",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

hgb_search.fit(hgb_X_train, hgb_y_train)

best_model_hgb = hgb_search.best_estimator_
hgb_proba = best_model_hgb.predict_proba(hgb_X_test)[:, 1]

print("HGB Best params:", hgb_search.best_params_)
print("HGB Best CV ROC AUC:", hgb_search.best_score_)
print("HGB Test ROC AUC:", roc_auc_score(hgb_y_test, hgb_proba))
print("HGB Test PR AUC:", average_precision_score(hgb_y_test, hgb_proba))

In [ ]:
results = pd.concat([results, pd.DataFrame([['HGB HyperParameter', roc_auc_score(hgb_y_test, hgb_proba), average_precision_score(hgb_y_test, hgb_proba)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
hgb_X_test_best = hgb_X_test.copy()
hgb_X_train_best = hgb_X_train.copy()

### Removing Unimportant features

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score
import pandas as pd

best_model_hgb.fit(hgb_X_train_best, hgb_y_train)

perm = permutation_importance(
    best_model_hgb,
    hgb_X_train_best,
    hgb_y_train,
    scoring="roc_auc",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

feat_importance = pd.Series(
    perm.importances_mean,
    index=hgb_X_train_best.columns
)

low_importance = feat_importance[feat_importance < 0.005].index

X_train_hgb = hgb_X_train_best.drop(columns=low_importance, errors="ignore")
X_test_hgb  = hgb_X_test_best.drop(columns=low_importance, errors="ignore")

best_model_hgb.fit(hgb_X_train, hgb_y_train)

y_proba_hgb = best_model_hgb.predict_proba(hgb_X_test)[:, 1]

print("HGB Test ROC AUC:", roc_auc_score(hgb_y_test, y_proba_hgb))
print("HGB Test PR AUC:", average_precision_score(hgb_y_test, y_proba_hgb))

In [ ]:
results = pd.concat([results, pd.DataFrame([['HGB Without Unimportant', roc_auc_score(hgb_y_test, y_proba_hgb), average_precision_score(hgb_y_test, y_proba_hgb)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
hgb_X_test_best = hgb_X_test.copy()
hgb_X_train_best = hgb_X_train.copy()

### Keeping only the top 20 features

In [ ]:
best_model_hgb.fit(hgb_X_train_best, hgb_y_train);

In [ ]:
from sklearn.inspection import permutation_importance
import pandas as pd

perm_hgb = permutation_importance(
    best_model_hgb,
    hgb_X_test_best,
    hgb_y_test,
    n_repeats=20,
    random_state=42,
    scoring="roc_auc",
    n_jobs=-1
)

hgb_importance = pd.DataFrame({
    "feature": hgb_X_test_best.columns,
    "importance": perm_hgb.importances_mean
}).sort_values("importance", ascending=False)

In [ ]:
top20_hgb = hgb_importance["feature"].head(20).tolist()

hgb_top20_result = evaluate_top_features(
    best_model_hgb,
    hgb_X_train_best,
    hgb_y_train,
    hgb_X_test_best,
    hgb_y_test,
    top20_hgb,
    "HGB Top 20"
)

print(
    hgb_top20_result["Model"],
    hgb_top20_result["ROC AUC"],
    hgb_top20_result["PR AUC"]
)

In [ ]:
hgb_X_test_best = hgb_top20_result["X_test_top"].copy()
hgb_X_train_best = hgb_top20_result["X_train_top"].copy()

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame([hgb_top20_result])[["Model", "ROC AUC", "PR AUC"]]
], ignore_index=True)

### Results

In [ ]:
results

#### SHAP

In [ ]:
best_model_hgb.fit(hgb_X_train_best, hgb_y_train);

In [ ]:
import shap
import numpy as np
import pandas as pd

import warnings

warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names"
)

import builtins

n_test = builtins.min(200, len(hgb_X_test_best))
n_train = builtins.min(100, len(hgb_X_train_best))

X_test_hgb_sample_df = hgb_X_test_best.sample(n=n_test, random_state=42)
X_background_df = hgb_X_train_best.sample(n=n_train, random_state=42)

X_background = np.asarray(X_background_df, dtype=np.float64)
X_sample = np.asarray(X_test_hgb_sample_df, dtype=np.float64)

explainer = shap.Explainer(best_model_hgb.predict_proba, X_background)

shap_values = explainer(X_sample)

shap_values_pos = shap_values.values[:, :, 1]
base_values_pos = shap_values.base_values[:, 1]

shap.summary_plot(shap_values_pos, X_test_hgb_sample_df, plot_type="bar")
shap.summary_plot(shap_values_pos, X_test_hgb_sample_df)

mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)

shap_importance = pd.DataFrame({
    "feature": X_test_hgb_sample_df.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

shap.initjs()
shap.force_plot(
    base_values_pos[0],
    shap_values_pos[0],
    X_test_hgb_sample_df.iloc[0]
)

### Check for Imbalance

In [ ]:
hgb_X_train_bal, hgb_y_train_bal = handle_imbalance(hgb_X_train_best, hgb_y_train)

In [ ]:
best_model_hgb.fit(hgb_X_train_bal, hgb_y_train)
y_proba_hgb = best_model_hgb.predict_proba(hgb_X_test_best)[:, 1]

print("HGB Test ROC AUC:", roc_auc_score(hgb_y_test, y_proba_hgb))
print("HGB Test PR AUC:", average_precision_score(hgb_y_test, y_proba_hgb))

### Threshold Analysis

In [ ]:
best_model_hgb.fit(hgb_X_train_best, hgb_y_train)
y_proba_hgb = best_model_hgb.predict_proba(hgb_X_test_best)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for t in thresholds:
    y_pred = (y_proba_hgb >= t).astype(int)

    rows.append({
        "Threshold": t,
        "Precision": precision_score(hgb_y_test, y_pred),
        "Recall": recall_score(hgb_y_test, y_pred),
        "F1": f1_score(hgb_y_test, y_pred)
    })

threshold_df_hgb = pd.DataFrame(rows)
threshold_df_hgb

### Extra Trees

In [ ]:
et_train_df = train_df.copy()
et_test_df = test_df.copy()

In [ ]:
et_X_train = et_train_df.drop(columns=[id_col, event_col])
et_y_train = et_train_df[event_col]

In [ ]:
et_X_test = et_test_df.drop(columns=[id_col, event_col])
et_y_test = et_test_df[event_col]

### Initial Training

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

et_default = ExtraTreesClassifier(random_state=42)

et_default.fit(et_X_train, et_y_train)

et_proba = et_default.predict_proba(et_X_test)[:, 1]

print("ET Test ROC AUC:", roc_auc_score(et_y_test, et_proba))
print("ET Test PR AUC:", average_precision_score(et_y_test, et_proba))

In [ ]:
results = pd.concat([results, pd.DataFrame([['ET Initial', roc_auc_score(et_y_test, et_proba), average_precision_score(et_y_test, et_proba)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Hyperparameter Tuning

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

et = ExtraTreesClassifier(random_state=42)

et = ExtraTreesClassifier(random_state=42, n_jobs=-1)

et_param_grid_v2 = [
    {
        "n_estimators": [400, 600, 800, 1000, 1200],
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 4, 6, 8, 10, 12],
        "max_leaf_nodes": [None, 15, 31, 63, 127],
        "min_samples_split": [2, 5, 10, 20, 40],
        "min_samples_leaf": [1, 2, 4, 8, 12],
        "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],
        "bootstrap": [False],
        "class_weight": [None, "balanced", "balanced_subsample"],
        "min_impurity_decrease": [0.0, 0.0001, 0.001, 0.005],
        "ccp_alpha": [0.0, 0.0001, 0.001, 0.005]
    },
    {
        "n_estimators": [400, 600, 800, 1000, 1200],
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 4, 6, 8, 10, 12],
        "max_leaf_nodes": [None, 15, 31, 63, 127],
        "min_samples_split": [2, 5, 10, 20, 40],
        "min_samples_leaf": [1, 2, 4, 8, 12],
        "max_features": ["sqrt", "log2", 0.3, 0.5, 0.7, 1.0],
        "bootstrap": [True],
        "max_samples": [0.6, 0.8, 1.0],
        "class_weight": [None, "balanced", "balanced_subsample"],
        "min_impurity_decrease": [0.0, 0.0001, 0.001, 0.005],
        "ccp_alpha": [0.0, 0.0001, 0.001, 0.005]
    }
]

et_search_roc = RandomizedSearchCV(
    estimator=et,
    param_distributions=et_param_grid_v2,
    n_iter=60,
    scoring="roc_auc",
    verbose=2,
    n_jobs=-1,
    random_state=42
)

et_search_roc.fit(et_X_train, et_y_train)

best_model_et = et_search_roc.best_estimator_
et_proba_roc = best_model_et.predict_proba(et_X_test)[:, 1]

print("ET ROC Best params:", et_search_roc.best_params_)
print("ET ROC Best CV ROC AUC:", et_search_roc.best_score_)
print("ET ROC Test ROC AUC:", roc_auc_score(et_y_test, et_proba_roc))
print("ET ROC Test PR AUC:", average_precision_score(et_y_test, et_proba_roc))

In [ ]:
results = pd.concat([results, pd.DataFrame([['ET Hyperparameter', roc_auc_score(et_y_test, et_proba_roc), average_precision_score(et_y_test, et_proba_roc)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

In [ ]:
et_X_test_best = et_X_test.copy()
et_X_train_best = et_X_train.copy()

### Removing unimportant features

In [ ]:
et_importance = pd.DataFrame({
    "feature": et_X_train_best.columns,
    "importance": best_model_et.feature_importances_
})

et_importance = et_importance.sort_values("importance", ascending=False)

low_importance_et = et_importance[
    et_importance["importance"] < 0.005
]["feature"]

In [ ]:
et_X_train = et_X_train_best.drop(columns=low_importance_et, errors="ignore")
et_X_test = et_X_test_best.drop(columns=low_importance_et, errors="ignore")

best_model_et.fit(et_X_train, et_y_train)

et_proba_roc = best_model_et.predict_proba(et_X_test)[:, 1]

print("ET ROC Test ROC AUC:", roc_auc_score(et_y_test, et_proba_roc))
print("ET ROC Test PR AUC:", average_precision_score(et_y_test, et_proba_roc))

In [ ]:
results = pd.concat([results, pd.DataFrame([['ET Without Unimportant', roc_auc_score(et_y_test, et_proba_roc), average_precision_score(et_y_test, et_proba_roc)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Keeping only the top 20 features

In [ ]:
best_model_et.fit(et_X_train_best, et_y_train);

In [ ]:
top20_et = et_importance.sort_values("importance", ascending=False)["feature"].head(20).tolist()

et_top20_result = evaluate_top_features(
    best_model_et,
    et_X_train_best,
    et_y_train,
    et_X_test_best,
    et_y_test,
    top20_et,
    "ET Top 20"
)

print(
    et_top20_result["Model"],
    et_top20_result["ROC AUC"],
    et_top20_result["PR AUC"]
)

In [ ]:
et_X_test_best = et_top20_result["X_test_top"].copy()
et_X_train_best = et_top20_result["X_train_top"].copy()

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame([et_top20_result])[["Model", "ROC AUC", "PR AUC"]]
], ignore_index=True)

### Results

In [ ]:
results

###

#### SHAP

In [ ]:
best_model_et.fit(et_X_train_best, et_y_train);

In [ ]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(best_model_et)

shap_exp = explainer(et_X_test_best)

if len(shap_exp.values.shape) == 3:
    # shape: (n_samples, n_features, n_classes)
    shap_values_pos = shap_exp.values[:, :, 1]
    base_value_pos = shap_exp.base_values[:, 1].mean()
else:
    # shape: (n_samples, n_features)
    shap_values_pos = shap_exp.values
    base_value_pos = np.mean(shap_exp.base_values)

shap.summary_plot(shap_values_pos, et_X_test_best, plot_type="bar")
shap.summary_plot(shap_values_pos, et_X_test_best)

mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
mean_abs_shap = np.ravel(mean_abs_shap)   # force 1D

shap_importance = pd.DataFrame({
    "feature": et_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

shap.initjs()
shap.force_plot(
    base_value_pos,
    shap_values_pos[0],
    et_X_test_best.iloc[0]
)

### Check for Imbalance

In [ ]:
et_X_train_bal, et_y_train_bal = handle_imbalance(et_X_train_best, et_y_train)

In [ ]:
best_model_et.fit(et_X_train_bal, et_y_train_bal)
et_proba_roc = best_model_et.predict_proba(et_X_test_best)[:, 1]

print("ET ROC Test ROC AUC:", roc_auc_score(et_y_test, et_proba_roc))
print("ET ROC Test PR AUC:", average_precision_score(et_y_test, et_proba_roc))

### Threshold Analysis

In [ ]:
best_model_et.fit(et_X_train_best, et_y_train)
et_proba_roc = best_model_et.predict_proba(et_X_test_best)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for t in thresholds:
    y_pred = (et_proba_roc >= t).astype(int)

    rows.append({
        "Threshold": t,
        "Precision": precision_score(et_y_test, y_pred),
        "Recall": recall_score(et_y_test, y_pred),
        "F1": f1_score(et_y_test, y_pred)
    })

threshold_df_et = pd.DataFrame(rows)
threshold_df_et

### LightGBM

In [ ]:
lgb_train_df = train_df.copy()
lgb_test_df = test_df.copy()

In [ ]:
lgb_X_train = lgb_train_df.drop(columns=[id_col, event_col])
lgb_y_train = lgb_train_df[event_col]

In [ ]:
lgb_X_test = lgb_test_df.drop(columns=[id_col, event_col])
lgb_y_test = lgb_test_df[event_col]

### Initial Training

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV

lgb_default = LGBMClassifier(random_state=42)

lgb_default.fit(lgb_X_train, lgb_y_train)

lgb_proba = lgb_default.predict_proba(lgb_X_test)[:, 1]

print("LGB ROC Test ROC AUC:", roc_auc_score(lgb_y_test, lgb_proba))
print("LGB ROC Test PR AUC:", average_precision_score(lgb_y_test, lgb_proba))

In [ ]:
results = pd.concat([results, pd.DataFrame([['LGB Initial', roc_auc_score(lgb_y_test, lgb_proba), average_precision_score(lgb_y_test, lgb_proba)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Hyperparameter Tuning

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score

scale_pos_weight = (lgb_y_train == 0).sum() / (lgb_y_train == 1).sum()

lgb = LGBMClassifier(
    objective="binary",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_param_dist = {
    "n_estimators": [300, 500, 800, 1200],
    "learning_rate": [0.005, 0.01, 0.02, 0.03],
    "num_leaves": [31, 63, 127, 255],
    "max_depth": [-1, 5, 7, 10],
    "min_child_samples": [10, 20, 50, 100],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5],
    "reg_lambda": [1, 3, 5, 10]
}

lgb_search_roc = RandomizedSearchCV(
    estimator=lgb,
    param_distributions=lgb_param_dist,
    n_iter=60,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

lgb_search_roc.fit(lgb_X_train, lgb_y_train)

best_model_lgb = lgb_search_roc.best_estimator_
lgb_proba_roc = best_model_lgb.predict_proba(lgb_X_test)[:, 1]

print("LGB ROC Best params:", lgb_search_roc.best_params_)
print("LGB ROC Best CV ROC AUC:", lgb_search_roc.best_score_)
print("LGB ROC Test ROC AUC:", roc_auc_score(lgb_y_test, lgb_proba_roc))
print("LGB ROC Test PR AUC:", average_precision_score(lgb_y_test, lgb_proba_roc))

In [ ]:
lgb_X_test_best = lgb_X_test.copy()
lgb_X_train_best = lgb_X_train.copy()

In [ ]:
results = pd.concat([results, pd.DataFrame([['LGB HyperParameter', roc_auc_score(lgb_y_test, lgb_proba_roc), average_precision_score(lgb_y_test, lgb_proba_roc)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Removing unimportant features

In [ ]:
lgb_importance = pd.DataFrame({
    "feature": lgb_X_train_best.columns,
    "importance": best_model_lgb.feature_importances_
})

lgb_importance = lgb_importance.sort_values("importance", ascending=False)

low_importance_lgb = lgb_importance[
    lgb_importance["importance"] < 0.005
]["feature"]

In [ ]:
lgb_X_train = lgb_X_train_best.drop(columns=low_importance_lgb, errors="ignore")
lgb_X_test = lgb_X_test_best.drop(columns=low_importance_lgb, errors="ignore")

best_model_lgb.fit(lgb_X_train, lgb_y_train)

y_proba_lgb = best_model_lgb.predict_proba(lgb_X_test)[:, 1]

print("Test ROC AUC:", roc_auc_score(lgb_y_test, y_proba_lgb))
print("Test PR AUC:", average_precision_score(lgb_y_test, y_proba_lgb))

In [ ]:
results = pd.concat([results, pd.DataFrame([['LGB Without Unimportant', roc_auc_score(lgb_y_test, y_proba_lgb), average_precision_score(lgb_y_test, y_proba_lgb)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Keeping only the top 20 features

In [ ]:
best_model_lgb.fit(lgb_X_train_best, lgb_y_train);

In [ ]:
lgb_importance = pd.DataFrame({
    "feature": lgb_X_train_best.columns,
    "importance": best_model_lgb.feature_importances_
}).sort_values("importance", ascending=False)

In [ ]:
top20_lgb = lgb_importance["feature"].head(20).tolist()

lgb_top20_result = evaluate_top_features(
    best_model_lgb,
    lgb_X_train_best,
    lgb_y_train,
    lgb_X_test_best,
    lgb_y_test,
    top20_lgb,
    "LGB Top 20"
)

print(
    lgb_top20_result["Model"],
    lgb_top20_result["ROC AUC"],
    lgb_top20_result["PR AUC"]
)

In [ ]:
results = pd.concat([
    results,
    pd.DataFrame([lgb_top20_result])[["Model", "ROC AUC", "PR AUC"]]
], ignore_index=True)

### Results

In [ ]:
results

#### SHAP

In [ ]:
best_model_lgb.fit(lgb_X_train_best, lgb_y_train);

In [ ]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(best_model_lgb)

shap_exp = explainer(lgb_X_test_best)

if len(shap_exp.values.shape) == 3:
    # shape: (n_samples, n_features, n_classes)
    shap_values_pos = shap_exp.values[:, :, 1]
    base_value_pos = shap_exp.base_values[:, 1].mean()
else:
    # shape: (n_samples, n_features)
    shap_values_pos = shap_exp.values
    base_value_pos = np.mean(shap_exp.base_values)

shap.summary_plot(shap_values_pos, lgb_X_test_best, plot_type="bar")
shap.summary_plot(shap_values_pos, lgb_X_test_best)

mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
mean_abs_shap = np.ravel(mean_abs_shap)

shap_importance = pd.DataFrame({
    "feature": lgb_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

# -----------------------------
# 6. local explanation
# -----------------------------
shap.initjs()
shap.force_plot(
    base_value_pos,
    shap_values_pos[0],
    lgb_X_test_best.iloc[0]
)

### No scale_pos_weight

In [ ]:
lgb_no_weight = LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgb_search_roc_no_weight = RandomizedSearchCV(
    estimator=lgb_no_weight,
    param_distributions=lgb_param_dist,
    n_iter=60,
    scoring="roc_auc",
    cv=5,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

lgb_search_roc_no_weight.fit(lgb_X_train_best, lgb_y_train)

best_model_lgb_no_weight = lgb_search_roc_no_weight.best_estimator_
lgb_proba_no_weight = best_model_lgb_no_weight.predict_proba(lgb_X_test_best)[:, 1]

print("NO WEIGHT Best CV ROC AUC:", lgb_search_roc_no_weight.best_score_)
print("NO WEIGHT Test ROC AUC:", roc_auc_score(lgb_y_test, lgb_proba_no_weight))
print("NO WEIGHT Test PR AUC:", average_precision_score(lgb_y_test, lgb_proba_no_weight))

In [ ]:
results = pd.concat([results, pd.DataFrame([['LGB No Weight', roc_auc_score(lgb_y_test, lgb_proba_no_weight), average_precision_score(lgb_y_test, lgb_proba_no_weight)]], 
                                           columns=['Model', 'ROC AUC', 'PR AUC'])], ignore_index=True)

### Results

In [ ]:
results

#### SHAP

In [ ]:
import shap
import numpy as np
import pandas as pd

explainer = shap.TreeExplainer(best_model_lgb_no_weight)

shap_exp = explainer(lgb_X_test_best)

if len(shap_exp.values.shape) == 3:
    # shape: (n_samples, n_features, n_classes)
    shap_values_pos = shap_exp.values[:, :, 1]
    base_value_pos = shap_exp.base_values[:, 1].mean()
else:
    # shape: (n_samples, n_features)
    shap_values_pos = shap_exp.values
    base_value_pos = np.mean(shap_exp.base_values)

shap.summary_plot(shap_values_pos, lgb_X_test_best, plot_type="bar")
shap.summary_plot(shap_values_pos, lgb_X_test_best)

mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
mean_abs_shap = np.ravel(mean_abs_shap)

shap_importance = pd.DataFrame({
    "feature": lgb_X_test_best.columns,
    "importance": mean_abs_shap
}).sort_values("importance", ascending=False)

print(shap_importance.head(15))

# -----------------------------
# 6. local explanation
# -----------------------------
shap.initjs()
shap.force_plot(
    base_value_pos,
    shap_values_pos[0],
    lgb_X_test_best.iloc[0]
)

### Check for Imbalance

In [ ]:
lgb_X_train_bal, lgb_y_train_bal = handle_imbalance(lgb_X_train_best, lgb_y_train)

In [ ]:
best_model_lgb_no_weight.fit(lgb_X_train_bal, lgb_y_train_bal)

y_proba_lgb = best_model_lgb_no_weight.predict_proba(lgb_X_test_best)[:, 1]

print("Test ROC AUC:", roc_auc_score(lgb_y_test, y_proba_lgb))
print("Test PR AUC:", average_precision_score(lgb_y_test, y_proba_lgb))

### Threshold Analysis

In [ ]:
best_model_lgb_no_weight.fit(lgb_X_train_best, lgb_y_train)

y_proba_lgb = best_model_lgb_no_weight.predict_proba(lgb_X_test_best)[:, 1]

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50]

rows = []

for t in thresholds:
    y_pred = (y_proba_lgb >= t).astype(int)

    rows.append({
        "Threshold": t,
        "Precision": precision_score(lgb_y_test, y_pred),
        "Recall": recall_score(lgb_y_test, y_pred),
        "F1": f1_score(lgb_y_test, y_pred)
    })

threshold_df_lgb = pd.DataFrame(rows)
threshold_df_lgb